# 02 Cleaning — F1 Race Strategy Intelligence


This notebook transforms 14 raw Ergast CSVs into three analysis-ready output files:
- `data/processed/master_fact.csv` — one row per race entry (~27K rows)
- `data/processed/constructor_season_kpis.csv` — one row per constructor-season
- `data/processed/circuit_strategy_profile.csv` — one row per circuit

## Section 1: Imports and Load
Load all 14 CSVs with `dtype=str` to preserve literal `\N` strings. Define constants used throughout the notebook.

In [1]:
import os
import warnings
import pandas as pd
import numpy as np

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

# ── Paths ──────────────────────────────────────────────────────────────────────
RAW_PATH       = '../data/raw/'
PROCESSED_PATH = '../data/processed/'
os.makedirs(PROCESSED_PATH, exist_ok=True)

# ── File registry ──────────────────────────────────────────────────────────────
FILES = [
    'results.csv', 'lap_times.csv', 'pit_stops.csv', 'qualifying.csv',
    'races.csv', 'circuits.csv', 'constructors.csv', 'drivers.csv',
    'status.csv', 'constructor_standings.csv', 'driver_standings.csv',
    'constructor_results.csv', 'sprint_results.csv', 'seasons.csv'
]

# ── Status classification constants ───────────────────────────────────────────
# statusId values that represent a classified finish (Finished or lapped cars)
FINISHED_STATUS = {'1','11','12','13','14','15','16','17','18','19'}

# ── Tracking dict for the final cleaning summary ──────────────────────────────
cleaning_log = []   # list of dicts: {step, description, records_affected, action}

# ── Load all files ─────────────────────────────────────────────────────────────
raw = {}
load_summary = []

for file in FILES:
    fp = os.path.join(RAW_PATH, file)
    if os.path.exists(fp):
        name = file.replace('.csv', '')
        raw[name] = pd.read_csv(fp, dtype=str)
        load_summary.append({
            'File': file,
            'Rows': len(raw[name]),
            'Cols': len(raw[name].columns)
        })
    else:
        print(f'[WARN] File not found: {fp}')

print('=== FILES LOADED ===')
display(pd.DataFrame(load_summary))
print(f'\nTotal files loaded: {len(raw)}')

=== FILES LOADED ===


,File,Rows,Cols
0,results.csv,27304,18
1,lap_times.csv,618766,6
2,pit_stops.csv,22193,7
3,qualifying.csv,11036,9
4,races.csv,1171,18
5,circuits.csv,78,9
6,constructors.csv,214,5
7,drivers.csv,865,9
8,status.csv,140,2
9,constructor_standings.csv,13664,7



Total files loaded: 14


## Section 2: Step 1: Global `\N` Replacement
Ergast uses the literal string `\N` to represent missing values.  
This must be converted to `pd.NA` **before any numeric operation**, otherwise casts will silently produce wrong values.

In [2]:
def replace_nulls(df: pd.DataFrame, name: str) -> int:
    """Replace all literal '\\N' strings in df with pd.NA and return replacement count."""
    before = int((df == r'\N').sum().sum())
    df.replace(r'\N', pd.NA, inplace=True)
    after  = int((df == r'\N').sum().sum())   # should always be 0
    print(f'[STEP 1] {name}: \\N replacements — before={before:,} → after={after}')
    return before

total_replaced = 0
for name, df in raw.items():
    total_replaced += replace_nulls(df, name)

print(f'\n[STEP 1] TOTAL \\N values replaced across all files: {total_replaced:,}')
cleaning_log.append({
    'Step': 1,
    'Description': 'Global \\N → pd.NA replacement',
    'Records Affected': total_replaced,
    'Action': 'df.replace(r\'\\N\', pd.NA, inplace=True) on all 14 DataFrames'
})

[STEP 1] results: \N replacements — before=123,882 → after=0


[STEP 1] lap_times: \N replacements — before=0 → after=0
[STEP 1] pit_stops: \N replacements — before=6 → after=0
[STEP 1] qualifying: \N replacements — before=12,093 → after=0
[STEP 1] races: \N replacements — before=11,477 → after=0
[STEP 1] circuits: \N replacements — before=0 → after=0
[STEP 1] constructors: \N replacements — before=0 → after=0
[STEP 1] drivers: \N replacements — before=1,559 → after=0
[STEP 1] status: \N replacements — before=0 → after=0
[STEP 1] constructor_standings: \N replacements — before=1 → after=0
[STEP 1] driver_standings: \N replacements — before=13 → after=0
[STEP 1] constructor_results: \N replacements — before=12,881 → after=0
[STEP 1] sprint_results: \N replacements — before=473 → after=0
[STEP 1] seasons: \N replacements — before=0 → after=0

[STEP 1] TOTAL \N values replaced across all files: 162,385


## Section 3: Step 2: Numeric Type Casting
Cast all columns that will participate in calculations to numeric types.  
`errors='coerce'` silently converts unparseable strings to `NaN` — we count and log those failures.

In [3]:
def cast_numeric(df: pd.DataFrame, df_name: str, cols: list) -> int:
    """Cast specified columns to numeric, log coercion failures, and return total NaNs introduced."""
    total_coerce_failures = 0
    for col in cols:
        if col not in df.columns:
            print(f'  [WARN] {df_name}.{col} not found — skipping')
            continue
        before_nulls = df[col].isna().sum()
        df[col] = pd.to_numeric(df[col], errors='coerce')
        after_nulls  = df[col].isna().sum()
        new_failures = after_nulls - before_nulls
        total_coerce_failures += new_failures
        if new_failures > 0:
            print(f'  [STEP 2] {df_name}.{col}: {new_failures} new NaN from coercion')
    return total_coerce_failures

CAST_MAP = {
    'results':   ['grid','position','positionOrder','points','laps',
                  'milliseconds','fastestLap','rank'],
    'pit_stops': ['milliseconds','stop','lap','duration'],
    'lap_times': ['milliseconds','lap','position'],
    'qualifying': ['position'],
    'circuits':  ['lat','lng','alt'],
    'races':     ['year','round'],
    'drivers':   [],   # no numeric cols needed beyond string keys
    'status':    [],
}

grand_total_failures = 0
print('[STEP 2] Numeric casting — reporting only columns with coercion failures:')
for df_name, cols in CAST_MAP.items():
    if df_name in raw and cols:
        n = cast_numeric(raw[df_name], df_name, cols)
        grand_total_failures += n

print(f'\n[STEP 2] Total coercion-induced NaN values across all casts: {grand_total_failures:,}')
cleaning_log.append({
    'Step': 2,
    'Description': 'Numeric type casting via pd.to_numeric(errors=coerce)',
    'Records Affected': grand_total_failures,
    'Action': 'Columns cast to float/int; unparseable values → NaN'
})

[STEP 2] Numeric casting — reporting only columns with coercion failures:
  [STEP 2] pit_stops.duration: 738 new NaN from coercion



[STEP 2] Total coercion-induced NaN values across all casts: 738


## Section 4: Step 3: Parse Qualifying Times → Milliseconds
Qualifying lap times are stored as strings like `"1:26.572"`.  
We parse them to milliseconds for arithmetic comparison, derive `best_q_ms` from the deepest round available, and compute each driver's gap to pole per race.

In [4]:
def parse_laptime(t) -> object:
    """Convert 'M:SS.mmm' lap time string to integer milliseconds; returns pd.NA on failure."""
    if pd.isna(t):
        return pd.NA
    try:
        parts = str(t).split(':')
        minutes = int(parts[0])
        seconds = float(parts[1])
        return int(minutes) * 60_000 + round(seconds * 1000)
    except Exception:
        return pd.NA

qualifying = raw['qualifying'].copy()

# ── Parse q1/q2/q3 to millisecond columns ─────────────────────────────────────
for q_col in ['q1', 'q2', 'q3']:
    ms_col = f'{q_col}_ms'
    qualifying[ms_col] = qualifying[q_col].apply(parse_laptime)
    # Ensure integer-compatible nullable int type
    qualifying[ms_col] = pd.to_numeric(qualifying[ms_col], errors='coerce')

# ── Derive best_q_ms: deepest round available for each driver ──────────────────
qualifying['best_q_ms'] = (
    qualifying['q3_ms']
    .combine_first(qualifying['q2_ms'])
    .combine_first(qualifying['q1_ms'])
)

# ── Log parse results ──────────────────────────────────────────────────────────
for q_col in ['q1', 'q2', 'q3']:
    ms_col = f'{q_col}_ms'
    parsed_ok  = qualifying[ms_col].notna().sum()
    parsed_na  = qualifying[ms_col].isna().sum()
    print(f'[STEP 3] {ms_col}: parsed successfully={parsed_ok:,}, NA={parsed_na:,}')

print(f'\n[STEP 3] best_q_ms sample (5 rows):')
display(qualifying[['raceId','driverId','q1_ms','q2_ms','q3_ms','best_q_ms']].dropna(
    subset=['best_q_ms']).head(5))

# ── Derive pole_ms per race (position == 1 → q3_ms of the pole sitter) ────────
# Cast position to numeric first for safe comparison
qualifying['position'] = pd.to_numeric(qualifying['position'], errors='coerce')

pole_times = (
    qualifying[qualifying['position'] == 1][['raceId', 'q3_ms']]
    .rename(columns={'q3_ms': 'pole_ms'})
)

# Some early-era races have no Q3 — fall back to best_q_ms of P1 driver
pole_best_fallback = (
    qualifying[qualifying['position'] == 1][['raceId', 'best_q_ms']]
    .rename(columns={'best_q_ms': 'pole_ms_fallback'})
)
pole_times = pole_times.merge(pole_best_fallback, on='raceId', how='left')
pole_times['pole_ms'] = pole_times['pole_ms'].combine_first(pole_times['pole_ms_fallback'])
pole_times = pole_times[['raceId', 'pole_ms']].drop_duplicates('raceId')

# ── Merge pole_ms back into qualifying ────────────────────────────────────────
qualifying = qualifying.merge(pole_times, on='raceId', how='left')
qualifying['qualifying_gap_ms'] = qualifying['best_q_ms'] - qualifying['pole_ms']

valid_gap_count = qualifying['qualifying_gap_ms'].notna().sum()
print(f'\n[STEP 3] Rows with valid qualifying_gap_ms: {valid_gap_count:,} / {len(qualifying):,}')

# Write cleaned qualifying back
raw['qualifying'] = qualifying

cleaning_log.append({
    'Step': 3,
    'Description': 'Parse qualifying times → milliseconds; derive qualifying_gap_ms',
    'Records Affected': valid_gap_count,
    'Action': 'parse_laptime() on q1/q2/q3; best_q_ms = q3.combine_first(q2).combine_first(q1); gap = best_q_ms - pole_ms'
})

[STEP 3] q1_ms: parsed successfully=10,873, NA=163
[STEP 3] q2_ms: parsed successfully=6,249, NA=4,787
[STEP 3] q3_ms: parsed successfully=3,893, NA=7,143

[STEP 3] best_q_ms sample (5 rows):


,raceId,driverId,q1_ms,q2_ms,q3_ms,best_q_ms
0,18,1,86572.0000,85187.0000,86714.0000,86714.0000
1,18,9,86103.0000,85315.0000,86869.0000,86869.0000
2,18,5,85664.0000,85452.0000,87079.0000,87079.0000
3,18,13,85994.0000,85691.0000,87178.0000,87178.0000
4,18,2,85960.0000,85518.0000,87236.0000,87236.0000



[STEP 3] Rows with valid qualifying_gap_ms: 10,875 / 11,036


## Section 5: Step 4: DNF Flags on Results
Create boolean indicator columns and classify the reason for each non-finish.  
Classification is based on the **status text** from `status.csv` (not hardcoded statusIds) so it remains correct if Ergast adds new codes.

In [5]:
results = raw['results'].copy()

# ── Boolean indicator columns ──────────────────────────────────────────────────
results['is_finisher']      = results['statusId'].astype(str).isin(FINISHED_STATUS)
results['is_dnf']           = ~results['is_finisher']
results['is_pitlane_start'] = (results['grid'] == 0)
results['is_podium']        = (results['positionOrder'] <= 3)
results['is_win']           = (results['positionOrder'] == 1)
results['is_pole']          = (results['grid'] == 1)

print('[STEP 4] Boolean flags created:')
for flag in ['is_finisher','is_dnf','is_pitlane_start','is_podium','is_win','is_pole']:
    print(f'  {flag}: True={results[flag].sum():,}, False={(~results[flag]).sum():,}')

# Write back
raw['results'] = results

cleaning_log.append({
    'Step': 4,
    'Description': 'DNF flags on results',
    'Records Affected': int(results['is_dnf'].sum()),
    'Action': 'Boolean indicators (is_finisher, is_dnf, etc.) added to results'
})

[STEP 4] Boolean flags created:
  is_finisher: True=15,494, False=11,810
  is_dnf: True=11,810, False=15,494
  is_pitlane_start: True=1,638, False=25,666
  is_podium: True=3,478, False=23,826
  is_win: True=1,155, False=26,149
  is_pole: True=1,162, False=26,142


## Section 6: Step 5: Pit Stop Aggregation
Aggregate pit stop records from the per-stop grain to a per-race-entry grain so they can be joined to `results`.

In [6]:
pit_stops = raw['pit_stops'].copy()

# Ensure numeric types (already cast in Step 2, but guard defensively)
for col in ['milliseconds', 'stop', 'lap', 'duration']:
    pit_stops[col] = pd.to_numeric(pit_stops[col], errors='coerce')

before_rows = len(pit_stops)

pit_agg = (
    pit_stops
    .groupby(['raceId', 'driverId'], as_index=False)
    .agg(
        stop_count     = ('stop',        'max'),   # max stop number = total stops
        avg_pit_ms     = ('milliseconds', 'mean'),
        total_pit_ms   = ('milliseconds', 'sum'),
        fastest_pit_ms = ('milliseconds', 'min'),
    )
)

after_rows = len(pit_agg)
print(f'[STEP 5] Pit stop aggregation: {before_rows:,} rows → {after_rows:,} driver-race entries')
print(f'         stop_count distribution:')
display(pit_agg['stop_count'].value_counts().sort_index().reset_index())

cleaning_log.append({
    'Step': 5,
    'Description': 'Pit stop aggregation (per-stop → per driver-race)',
    'Records Affected': before_rows,
    'Action': f'groupby([raceId, driverId]) → {after_rows:,} rows with stop_count, avg/total/fastest pit ms'
})

[STEP 5] Pit stop aggregation: 22,193 rows → 11,307 driver-race entries
         stop_count distribution:


,stop_count,count
0,1,3740
1,2,5059
2,3,1912
3,4,423
4,5,132
5,6,38
6,7,3


## Section 7: Step 6: Lap Times Summary
Summarise the 618K+ lap-time rows to one row per driver-race for joining.  
Scoped to **2010+** as specified (pit density KPI scope), but the summary table itself is unrestricted; the 2010 filter is applied only where noted.

In [7]:
lap_times = raw['lap_times'].copy()
races_ref = raw['races'][['raceId','year']].copy()
races_ref['year'] = pd.to_numeric(races_ref['year'], errors='coerce')

# Merge year onto lap_times to allow year-based scoping
lap_times = lap_times.merge(races_ref, on='raceId', how='left')

# Filter to 2010+
lap_times_filtered = lap_times[lap_times['year'] >= 2010].copy()

before_rows = len(lap_times)
after_rows  = len(lap_times_filtered)
print(f'[STEP 6] Lap times rows: all={before_rows:,} | 2010+={after_rows:,}')

lap_summary = (
    lap_times_filtered
    .groupby(['raceId', 'driverId'], as_index=False)
    .agg(
        fastest_lap_ms = ('milliseconds', 'min'),
        lap_count      = ('lap',          'count'),
        lap_time_std   = ('milliseconds', 'std'),
    )
)

print(f'[STEP 6] Lap summary: {len(lap_summary):,} driver-race rows | year scope: 2010+')
print(f'         avg fastest_lap_ms: {lap_summary["fastest_lap_ms"].mean():,.1f} ms')

cleaning_log.append({
    'Step': 6,
    'Description': 'Lap times summary (2010+, per driver-race)',
    'Records Affected': after_rows,
    'Action': f'groupby([raceId, driverId]) → {len(lap_summary):,} rows with fastest/count/std lap ms'
})

[STEP 6] Lap times rows: all=618,766 | 2010+=372,493
[STEP 6] Lap summary: 6,767 driver-race rows | year scope: 2010+
         avg fastest_lap_ms: 92,107.1 ms


## Section 8: Step 7: Master Merge
Build `master_fact` by sequentially left-joining dimension and aggregated tables onto `results`.  
Row count is asserted equal to 27,304 after every join.

In [8]:
EXPECTED_ROWS = 27_304   # immutable constant — asserted after every join

def assert_rows(df: pd.DataFrame, step_label: str, expected: int = EXPECTED_ROWS) -> None:
    """Assert DataFrame row count matches expected value, else print a warning."""
    if len(df) != expected:
        print(f'[WARN] {step_label}: expected {expected:,} rows but got {len(df):,}')
    else:
        print(f'[STEP 7] {step_label}: row count OK → {len(df):,}')

# ── Start with results (already enriched with boolean flags) ──────────────────
master = raw['results'].copy()
print(f'[STEP 7] Starting master shape: {master.shape}')

# ── Join 1: races ─────────────────────────────────────────────────────────────
races_cols = raw['races'][['raceId','year','round','circuitId','name']].rename(
    columns={'name': 'race_name'}
)
master = master.merge(races_cols, on='raceId', how='left')
assert_rows(master, 'After joining races')

# ── Join 2: circuits ──────────────────────────────────────────────────────────
circuits_cols = raw['circuits'][['circuitId','name','location','country','lat','lng']].rename(
    columns={'name': 'circuit_name'}
)
master = master.merge(circuits_cols, on='circuitId', how='left')
assert_rows(master, 'After joining circuits')

# ── Join 3: constructors ──────────────────────────────────────────────────────
constructors_cols = raw['constructors'][['constructorId','name','nationality']].rename(
    columns={'name': 'constructor_name', 'nationality': 'constructor_nationality'}
)
master = master.merge(constructors_cols, on='constructorId', how='left')
assert_rows(master, 'After joining constructors')

# ── Join 4: drivers ───────────────────────────────────────────────────────────
drivers_cols = raw['drivers'][['driverId','forename','surname','nationality']].rename(
    columns={'nationality': 'driver_nationality'}
)
master = master.merge(drivers_cols, on='driverId', how='left')
assert_rows(master, 'After joining drivers')

# ── Join 5: pit_agg ───────────────────────────────────────────────────────────
pit_agg['raceId']   = pit_agg['raceId'].astype(str)
pit_agg['driverId'] = pit_agg['driverId'].astype(str)
master = master.merge(
    pit_agg[['raceId','driverId','stop_count','avg_pit_ms','total_pit_ms','fastest_pit_ms']],
    on=['raceId','driverId'], how='left'
)
assert_rows(master, 'After joining pit_agg')

# ── Join 6: qualifying ────────────────────────────────────────────────────────
qual_cols = raw['qualifying'][['raceId','driverId','best_q_ms','qualifying_gap_ms',
                               'q1_ms','q2_ms','q3_ms']]
qual_cols = qual_cols.copy()
qual_cols['raceId']   = qual_cols['raceId'].astype(str)
qual_cols['driverId'] = qual_cols['driverId'].astype(str)
master = master.merge(qual_cols, on=['raceId','driverId'], how='left')
assert_rows(master, 'After joining qualifying')

# ── Join 7: lap_summary ───────────────────────────────────────────────────────
lap_summary['raceId']   = lap_summary['raceId'].astype(str)
lap_summary['driverId'] = lap_summary['driverId'].astype(str)
master = master.merge(
    lap_summary[['raceId','driverId','fastest_lap_ms','lap_count','lap_time_std']],
    on=['raceId','driverId'], how='left'
)
assert_rows(master, 'After joining lap_summary')

# ── Join 8: status ────────────────────────────────────────────────────────────
status_ref = raw['status'][['statusId','status']]
master['statusId'] = master['statusId'].astype(str)
status_ref['statusId'] = status_ref['statusId'].astype(str)
master = master.merge(status_ref, on='statusId', how='left')
assert_rows(master, 'After joining status')

# ── DNF category classification using status TEXT ─────────────────────────────
MECHANICAL_KEYWORDS = ['engine','gearbox','transmission','hydraulic','electrical',
                       'brakes','brake','suspension','throttle','electronics',
                       'differential','clutch','fuel','oil','water','turbo',
                       'exhaust','overheating','fire','mechanical','power unit',
                       'driveshaft','drivetrain']
ACCIDENT_KEYWORDS   = ['collision','accident','spun off','spin','crash','damage',
                       'puncture','tyre','tire','wheel']
DNQ_KEYWORDS        = ['did not qualify','did not prequalify','not classified',
                       'excluded','withdrawn','disqualified']

def classify_dnf(row) -> str:
    """Return a DNF category string based on the status text of a result row."""
    if row['is_finisher']:
        return 'Finished'
    status_lower = str(row['status']).lower() if pd.notna(row['status']) else ''
    for kw in DNQ_KEYWORDS:
        if kw in status_lower: return 'DNQ'
    for kw in MECHANICAL_KEYWORDS:
        if kw in status_lower: return 'Mechanical'
    for kw in ACCIDENT_KEYWORDS:
        if kw in status_lower: return 'Accident'
    return 'Other'

master['dnf_category'] = master.apply(classify_dnf, axis=1)

print(f'\n[STEP 7] Final master shape: {master.shape}')
print(f'[STEP 7] Columns: {list(master.columns)}')

cleaning_log.append({
    'Step': 7,
    'Description': 'Master merge (8 sequential left joins on results)',
    'Records Affected': len(master),
    'Action': 'Left join: races, circuits, constructors, drivers, pit_agg, qualifying, lap_summary, status; dnf_category derived'
})

[STEP 7] Starting master shape: (27304, 24)
[STEP 7] After joining races: row count OK → 27,304
[STEP 7] After joining circuits: row count OK → 27,304


[STEP 7] After joining constructors: row count OK → 27,304
[STEP 7] After joining drivers: row count OK → 27,304
[STEP 7] After joining pit_agg: row count OK → 27,304
[STEP 7] After joining qualifying: row count OK → 27,304


[STEP 7] After joining lap_summary: row count OK → 27,304


[STEP 7] After joining status: row count OK → 27,304



[STEP 7] Final master shape: (27304, 52)
[STEP 7] Columns: ['resultId', 'raceId', 'driverId', 'constructorId', 'number', 'grid', 'position', 'positionText', 'positionOrder', 'points', 'laps', 'time', 'milliseconds', 'fastestLap', 'rank', 'fastestLapTime', 'fastestLapSpeed', 'statusId', 'is_finisher', 'is_dnf', 'is_pitlane_start', 'is_podium', 'is_win', 'is_pole', 'year', 'round', 'circuitId', 'race_name', 'circuit_name', 'location', 'country', 'lat', 'lng', 'constructor_name', 'constructor_nationality', 'forename', 'surname', 'driver_nationality', 'stop_count', 'avg_pit_ms', 'total_pit_ms', 'fastest_pit_ms', 'best_q_ms', 'qualifying_gap_ms', 'q1_ms', 'q2_ms', 'q3_ms', 'fastest_lap_ms', 'lap_count', 'lap_time_std', 'status', 'dnf_category']


## Section 9: Step 8: Derive KPI Columns on Master
Compute derived columns directly on the master DataFrame.

In [9]:
# ── Ensure year is numeric (may still be object if not in raw['races'] numeric cast) ──
master['year'] = pd.to_numeric(master['year'], errors='coerce')

# ── KPI1: Grid-to-Finish Delta ────────────────────────────────────────────────
# Only valid where: not a pit-lane start AND driver is classified as a finisher
valid_delta_mask = (~master['is_pitlane_start']) & (master['is_finisher'])
master['grid_to_finish_delta'] = np.where(
    valid_delta_mask,
    master['grid'] - master['positionOrder'],
    np.nan
)

delta_count = master['grid_to_finish_delta'].notna().sum()
print(f'[STEP 8] grid_to_finish_delta: {delta_count:,} valid values '
      f'(mean={master["grid_to_finish_delta"].mean():.3f})')

# ── driver_name ───────────────────────────────────────────────────────────────
master['driver_name'] = master['forename'].fillna('') + ' ' + master['surname'].fillna('')
master['driver_name'] = master['driver_name'].str.strip()

# ── era classification ────────────────────────────────────────────────────────
def classify_era(y) -> str:
    """Map a season year to an F1 engine era label."""
    if pd.isna(y):
        return 'Unknown'
    y = int(y)
    if y < 1983:
        return 'Pre-Turbo'
    elif y < 1989:
        return 'Turbo'
    elif y < 2014:
        return 'V10/V8'
    else:
        return 'Hybrid'

master['era'] = master['year'].apply(classify_era)

print(f'[STEP 8] era distribution:')
display(master['era'].value_counts().reset_index())
print(f'[STEP 8] driver_name sample: {master["driver_name"].head(5).tolist()}')

cleaning_log.append({
    'Step': 8,
    'Description': 'Derived KPI columns on master',
    'Records Affected': int(valid_delta_mask.sum()),
    'Action': 'grid_to_finish_delta, driver_name, era added to master'
})

[STEP 8] grid_to_finish_delta: 15,422 valid values (mean=3.134)
[STEP 8] era distribution:


,era,count
0,V10/V8,10323
1,Pre-Turbo,9224
2,Hybrid,5171
3,Turbo,2586


[STEP 8] driver_name sample: ['Lewis Hamilton', 'Nick Heidfeld', 'Nico Rosberg', 'Fernando Alonso', 'Heikki Kovalainen']


## Section 10: Step 9: Build Aggregated Output Tables
### Table A — `constructor_season_kpis.csv`
One row per constructor-season with all strategic KPIs rolled up.

In [10]:
# ── Helper: safe division to avoid ZeroDivisionError in agg ───────────────────
def safe_div(num, denom):
    """Return num/denom or 0 if denom is zero."""
    return num / denom if denom > 0 else 0.0

# ── Aggregate per constructor-season ─────────────────────────────────────────
grp = master.groupby(['constructorId', 'constructor_name', 'year'])

constructor_kpis = grp.apply(
    lambda g: pd.Series({
        'total_points':      g['points'].sum(),
        'races_entered':     len(g),
        'points_efficiency': safe_div(g['points'].sum(), len(g)),
        'podium_rate':       safe_div(g['is_podium'].sum(), len(g)),
        'win_rate':          safe_div(g['is_win'].sum(), len(g)),
        'pole_count':        int(g['is_pole'].sum()),
        'pole_to_win_rate':  safe_div(
                                 (g['is_pole'] & g['is_win']).sum(),
                                 g['is_pole'].sum()
                             ),
        'dnf_rate':          safe_div(g['is_dnf'].sum(), len(g)),
        'avg_grid':          g.loc[g['grid'] > 0, 'grid'].mean(),
        'avg_delta':         g['grid_to_finish_delta'].mean(),
        'points_volatility': g['points'].std(),
        'avg_pit_ms':        g['avg_pit_ms'].mean(),       # NaN for pre-pit era
        'avg_stop_count':    g['stop_count'].mean(),       # NaN for pre-pit era
    })
).reset_index()

print(f'[STEP 9A] constructor_season_kpis shape: {constructor_kpis.shape}')
display(constructor_kpis.head(5))

cleaning_log.append({
    'Step': '9A',
    'Description': 'Constructor-season KPI aggregation',
    'Records Affected': len(constructor_kpis),
    'Action': '13 KPI columns grouped by constructorId × year'
})

[STEP 9A] constructor_season_kpis shape: (1132, 16)


,constructorId,constructor_name,year,total_points,races_entered,points_efficiency,podium_rate,win_rate,pole_count,pole_to_win_rate,dnf_rate,avg_grid,avg_delta,points_volatility,avg_pit_ms,avg_stop_count
0,1,McLaren,1968,0.0000,2.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.5000,10.5000,3.0000,0.0000,NaN,NaN
1,1,McLaren,1971,13.0000,27.0000,0.4815,0.0370,0.0000,0.0000,0.0000,0.5556,13.9615,5.9167,1.1222,NaN,NaN
2,1,McLaren,1972,66.0000,25.0000,2.6400,0.4400,0.0400,1.0000,0.0000,0.1600,6.4800,2.1905,2.5146,NaN,NaN
3,1,McLaren,1973,68.0000,35.0000,1.9429,0.2286,0.0857,1.0000,0.0000,0.2571,5.5143,-0.3462,2.7434,NaN,NaN
4,1,McLaren,1974,87.0000,46.0000,1.8913,0.2174,0.0870,2.0000,1.0000,0.2609,10.0000,4.3235,2.8459,NaN,NaN


### Table B — `circuit_strategy_profile.csv`
One row per circuit using classified finishers from 2010 onwards.

In [11]:
# ── Filter: finishers only, 2010+ ─────────────────────────────────────────────
circuit_df = master[
    (master['is_finisher'] == True) &
    (master['year'] >= 2010)
].copy()

print(f'[STEP 9B] Circuit profile input rows (finishers 2010+): {len(circuit_df):,}')

# ── Best stop strategy per circuit ────────────────────────────────────────────
# Find stop_count associated with best (lowest) avg finishing position per circuit
stop_strategy = (
    circuit_df.dropna(subset=['stop_count'])
    .groupby(['circuitId', 'stop_count'], as_index=False)['positionOrder']
    .mean()
    .rename(columns={'positionOrder': 'avg_pos_for_stop_count'})
)
# Pick the stop_count row with the lowest avg position per circuit
best_stop = (
    stop_strategy
    .sort_values('avg_pos_for_stop_count')
    .groupby('circuitId', as_index=False)
    .first()
    [['circuitId','stop_count']]
    .rename(columns={'stop_count': 'best_strategy_stops'})
)

# ── Per-stop-count average finishing positions ─────────────────────────────────
avg_1stop = (
    circuit_df[circuit_df['stop_count'] == 1]
    .groupby('circuitId')['positionOrder'].mean()
    .reset_index()
    .rename(columns={'positionOrder': 'avg_1stop_position'})
)
avg_2stop = (
    circuit_df[circuit_df['stop_count'] == 2]
    .groupby('circuitId')['positionOrder'].mean()
    .reset_index()
    .rename(columns={'positionOrder': 'avg_2stop_position'})
)

# ── Main circuit aggregation ───────────────────────────────────────────────────
circuit_profile = (
    circuit_df
    .groupby(['circuitId', 'circuit_name', 'country'], as_index=False)
    .agg(
        total_races          = ('raceId',               'nunique'),
        avg_delta            = ('grid_to_finish_delta', 'mean'),
        avg_qualifying_gap   = ('qualifying_gap_ms',    'mean'),
        lap_time_variance    = ('lap_time_std',         'mean'),
    )
)

# ── Merge derived columns ──────────────────────────────────────────────────────
circuit_profile = circuit_profile.merge(best_stop, on='circuitId', how='left')
circuit_profile = circuit_profile.merge(avg_1stop,  on='circuitId', how='left')
circuit_profile = circuit_profile.merge(avg_2stop,  on='circuitId', how='left')

# ── Overtaking score: rank lap_time_variance into High/Medium/Low ──────────────
# Higher variance = more position change opportunity = higher overtaking score
circuit_profile['variance_rank'] = circuit_profile['lap_time_variance'].rank(
    method='first', na_option='bottom'
)
n_circuits = circuit_profile['variance_rank'].notna().sum()
def rank_to_tier(r, n) -> str:
    """Map a rank to a High/Medium/Low overtaking tier."""
    if pd.isna(r):
        return 'Unknown'
    pct = r / n
    if pct >= 0.67:
        return 'High'
    elif pct >= 0.33:
        return 'Medium'
    else:
        return 'Low'

circuit_profile['overtaking_score'] = circuit_profile['variance_rank'].apply(
    lambda r: rank_to_tier(r, n_circuits)
)
circuit_profile.drop(columns=['variance_rank'], inplace=True)

print(f'[STEP 9B] circuit_strategy_profile shape: {circuit_profile.shape}')
print(f'[STEP 9B] overtaking_score distribution:')
display(circuit_profile['overtaking_score'].value_counts().reset_index())
display(circuit_profile.head(5))

cleaning_log.append({
    'Step': '9B',
    'Description': 'Circuit strategy profile aggregation (finishers 2010+)',
    'Records Affected': len(circuit_profile),
    'Action': 'Grouped by circuitId; best stop strategy, avg positions, overtaking score derived'
})

[STEP 9B] Circuit profile input rows (finishers 2010+): 5,745
[STEP 9B] circuit_strategy_profile shape: (35, 11)
[STEP 9B] overtaking_score distribution:


,overtaking_score,count
0,Medium,12
1,High,12
2,Low,11


,circuitId,circuit_name,country,total_races,avg_delta,avg_qualifying_gap,lap_time_variance,best_strategy_stops,avg_1stop_position,avg_2stop_position,overtaking_score
0,1,Albert Park Grand Prix Circuit,Australia,15,2.1818,2327.7143,36718.8770,1.0000,6.9500,8.5802,Medium
1,10,Hockenheimring,Germany,6,1.5278,7.1321,6592.0134,5.0000,7.4444,10.7419,Low
2,11,Hungaroring,Hungary,16,1.2105,2059.8287,13819.0516,6.0000,9.6364,9.5000,Medium
3,12,Valencia Street Circuit,Spain,3,1.7778,2135.5397,9526.7975,1.0000,9.5556,11.5652,Medium
4,13,Circuit de Spa-Francorchamps,Belgium,16,1.7572,2669.7590,21565.9171,1.0000,8.5273,8.9869,Medium


## Section 11: Export
Write the three processed files to `data/processed/`. Print final shapes and column inventory.

In [12]:
# ── Export ─────────────────────────────────────────────────────────────────────
master_path           = os.path.join(PROCESSED_PATH, 'master_fact.csv')
constructor_kpis_path = os.path.join(PROCESSED_PATH, 'constructor_season_kpis.csv')
circuit_profile_path  = os.path.join(PROCESSED_PATH, 'circuit_strategy_profile.csv')

master.to_csv(master_path,           index=False)
constructor_kpis.to_csv(constructor_kpis_path, index=False)
circuit_profile.to_csv(circuit_profile_path,   index=False)

print('=== EXPORT COMPLETE ===')
print(f'master_fact.csv            → {master.shape}')
print(f'constructor_season_kpis.csv → {constructor_kpis.shape}')
print(f'circuit_strategy_profile.csv → {circuit_profile.shape}')

print('\n=== master_fact.csv COLUMNS ===')
for i, col in enumerate(master.columns, 1):
    print(f'  {i:02d}. {col}')

=== EXPORT COMPLETE ===
master_fact.csv            → (27304, 55)
constructor_season_kpis.csv → (1132, 16)
circuit_strategy_profile.csv → (35, 11)

=== master_fact.csv COLUMNS ===
  01. resultId
  02. raceId
  03. driverId
  04. constructorId
  05. number
  06. grid
  07. position
  08. positionText
  09. positionOrder
  10. points
  11. laps
  12. time
  13. milliseconds
  14. fastestLap
  15. rank
  16. fastestLapTime
  17. fastestLapSpeed
  18. statusId
  19. is_finisher
  20. is_dnf
  21. is_pitlane_start
  22. is_podium
  23. is_win
  24. is_pole
  25. year
  26. round
  27. circuitId
  28. race_name
  29. circuit_name
  30. location
  31. country
  32. lat
  33. lng
  34. constructor_name
  35. constructor_nationality
  36. forename
  37. surname
  38. driver_nationality
  39. stop_count
  40. avg_pit_ms
  41. total_pit_ms
  42. fastest_pit_ms
  43. best_q_ms
  44. qualifying_gap_ms
  45. q1_ms
  46. q2_ms
  47. q3_ms
  48. fastest_lap_ms
  49. lap_count
  50. lap_time_std
  51. s

## Section 12: Cleaning Summary
A consolidated table of every transformation step: what was done, how many records were affected, and the action taken.

In [13]:
summary_df = pd.DataFrame(cleaning_log)
summary_df.columns = ['Step', 'Description', 'Records Affected', 'Action']

print('=' * 80)
print('CLEANING SUMMARY')
print('=' * 80)
display(summary_df)

print(f'\nFinal master_fact row count : {len(master):,}')
print(f'Final master_fact col count : {len(master.columns)}')
print(f'Final constructor_kpis rows : {len(constructor_kpis):,}')
print(f'Final circuit_profile rows  : {len(circuit_profile):,}')


CLEANING SUMMARY


,Step,Description,Records Affected,Action
0,1,Global \N → pd.NA replacement,162385,"df.replace(r'\N', pd.NA, inplace=True) on all ..."
1,2,Numeric type casting via pd.to_numeric(errors=...,738,Columns cast to float/int; unparseable values ...
2,3,Parse qualifying times → milliseconds; derive ...,10875,parse_laptime() on q1/q2/q3; best_q_ms = q3.co...
3,4,DNF flags on results,11810,"Boolean indicators (is_finisher, is_dnf, etc.)..."
4,5,Pit stop aggregation (per-stop → per driver-race),22193,"groupby([raceId, driverId]) → 11,307 rows with..."
5,6,"Lap times summary (2010+, per driver-race)",372493,"groupby([raceId, driverId]) → 6,767 rows with ..."
6,7,Master merge (8 sequential left joins on results),27304,"Left join: races, circuits, constructors, driv..."
7,8,Derived KPI columns on master,15438,"grid_to_finish_delta, driver_name, era added t..."
8,9A,Constructor-season KPI aggregation,1132,13 KPI columns grouped by constructorId × year
9,9B,Circuit strategy profile aggregation (finisher...,35,"Grouped by circuitId; best stop strategy, avg ..."



Final master_fact row count : 27,304
Final master_fact col count : 55
Final constructor_kpis rows : 1,132
Final circuit_profile rows  : 35
